# Data Cleaning

## Imports

In [1]:
import re
import html
import time
import nltk
import spacy
import pandas as pd

from tqdm import tqdm
tqdm.pandas()

from functools import partial
from emot.emo_unicode import EMOTICONS_EMO
from multiprocessing import Pool, cpu_count
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [2]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/swar/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

## Load Raw Data

In [3]:
df = pd.read_csv("../data/raw/training.1600000.processed.noemoticon.csv",
                encoding="latin-1",header=None,
                names=["sentiment", "id", "date", "query", "user", "tweet"])

In [4]:
# Dropping unnsecery cols
df.drop(columns=["user", "query", "date"], inplace=True)
df.head()

,sentiment,id,tweet
0,0,1467810369,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,is upset that he can't update his Facebook by ...
2,0,1467810917,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,my whole body feels itchy and like its on fire
4,0,1467811193,"@nationwideclass no, it's not behaving at all...."


## 3. Cleaning Pipeline

In [5]:
test_tweets = {
    0: "Bought yarn to make daughters &quot;skirt&quot; to wear with leggings. She's fun to make things for ;) I think I'll write this pattern out ",
    1: "@Impala_Guy REALLY - thatï¿½s cool  Iï¿½m an Horror Movie Fan too.....my best girlfriend gave me the nickname SPOOKY because of that ))",
    2: "is a perfectionist and has been still working here and there on her #Debian layouts, but now it's time for /dev/shower &amp; /dev/cup/tea. ",
    3: "@ddlovato and @TraceCyrus .... YUCKKKKKKKK!! eww.. what is she thinking!? she is soo beautiful and could do SO MUCHHH BETTER!!!!   ", 
    4: "have heart whaaaaat? ",
    5: "@beckharkin haha. im gooooood thanx  but very bored :\\ twitter. not facebook. myspace. anything. footy. btw, congrats on the win  hahaha",
    6: "Day off tomorrow, so gym, shopping and holiday searching - still need a travel buddy, so if you know anyone interested... &lt;/desperate&gt; ",
    7: "http://bit.ly/mAxQY  ~ ok hope this link works  ",
    8: "cÃ¡i yume nÃ³ lag vÃ£i (...yobanbe thÃ¬ nÃ³ máº¥t pass....dis...dek cÃ³ cÃ¡i quáº§n gÃ¬ Äá» chÆ¡i ",
    9: "@bdiibusiness I NO!!!!!! WE ARE SOOOOOOO KOOOOOOL!!!!!!!!!!!  and ill take ur word for it not being my kind of thing ",
    10: "@MrWize u talking like u know where the fuck i live 1700 walkus ct district heights BITCH do you ",
}

In [6]:
def show_step(tweets_dict, step_func, step_name):
    print("=" * 50)
    print(f" STEP: {step_name}")
    print("=" * 50)
    result = {}
    for idx, tweet in tweets_dict.items():
        cleaned = step_func(tweet)
        if cleaned != tweet:
            print(f"\n[Tweet {idx}]")
            print(f"  BEFORE: {tweet}")
            print(f"  AFTER : {cleaned}")
            print("-" * 40)
        result[idx] = cleaned
    return result

### 3.1 HTML Entity Fixing

In [7]:
tweets = show_step(test_tweets, lambda text: html.unescape(text), "HTML Entity Fixing")

 STEP: HTML Entity Fixing

[Tweet 0]
  BEFORE: Bought yarn to make daughters &quot;skirt&quot; to wear with leggings. She's fun to make things for ;) I think I'll write this pattern out 
  AFTER : Bought yarn to make daughters "skirt" to wear with leggings. She's fun to make things for ;) I think I'll write this pattern out 
----------------------------------------

[Tweet 2]
  BEFORE: is a perfectionist and has been still working here and there on her #Debian layouts, but now it's time for /dev/shower &amp; /dev/cup/tea. 
  AFTER : is a perfectionist and has been still working here and there on her #Debian layouts, but now it's time for /dev/shower & /dev/cup/tea. 
----------------------------------------

[Tweet 6]
  BEFORE: Day off tomorrow, so gym, shopping and holiday searching - still need a travel buddy, so if you know anyone interested... &lt;/desperate&gt; 
  AFTER : Day off tomorrow, so gym, shopping and holiday searching - still need a travel buddy, so if you know anyone int

### 3.2 Remove URLs

In [8]:
def remove_urls(text):
    text = re.sub(r"http[s]?://\S+", "", text)
    text = re.sub(r"www\.\S+", "", text)
    return text

tweets = show_step(tweets, remove_urls, "Remove URLs")

 STEP: Remove URLs

[Tweet 7]
  BEFORE: http://bit.ly/mAxQY  ~ ok hope this link works  
  AFTER :   ~ ok hope this link works  
----------------------------------------


### 3.3 Remove Mentions

In [9]:
tweets = show_step(tweets, lambda text: re.sub(r"@\w+", "", text), "Remove Mentions")

 STEP: Remove Mentions

[Tweet 1]
  BEFORE: @Impala_Guy REALLY - thatï¿½s cool  Iï¿½m an Horror Movie Fan too.....my best girlfriend gave me the nickname SPOOKY because of that ))
  AFTER :  REALLY - thatï¿½s cool  Iï¿½m an Horror Movie Fan too.....my best girlfriend gave me the nickname SPOOKY because of that ))
----------------------------------------

[Tweet 3]
  BEFORE: @ddlovato and @TraceCyrus .... YUCKKKKKKKK!! eww.. what is she thinking!? she is soo beautiful and could do SO MUCHHH BETTER!!!!   
  AFTER :  and  .... YUCKKKKKKKK!! eww.. what is she thinking!? she is soo beautiful and could do SO MUCHHH BETTER!!!!   
----------------------------------------

[Tweet 5]
  BEFORE: @beckharkin haha. im gooooood thanx  but very bored :\ twitter. not facebook. myspace. anything. footy. btw, congrats on the win  hahaha
  AFTER :  haha. im gooooood thanx  but very bored :\ twitter. not facebook. myspace. anything. footy. btw, congrats on the win  hahaha
----------------------------------

### 3.4 Remove Hashtag Symbols

In [10]:
tweets = show_step(tweets, lambda text: re.sub(r"#", "", text), "Remove Hashtag")

 STEP: Remove Hashtag

[Tweet 2]
  BEFORE: is a perfectionist and has been still working here and there on her #Debian layouts, but now it's time for /dev/shower & /dev/cup/tea. 
  AFTER : is a perfectionist and has been still working here and there on her Debian layouts, but now it's time for /dev/shower & /dev/cup/tea. 
----------------------------------------


### 3.5 Cap Elongated Words

In [11]:
tweets = show_step(tweets, lambda text: re.sub(r"(.)\1{2,}", r"\1\1", text), "Cap Elongated Words")

 STEP: Cap Elongated Words

[Tweet 1]
  BEFORE:  REALLY - thatï¿½s cool  Iï¿½m an Horror Movie Fan too.....my best girlfriend gave me the nickname SPOOKY because of that ))
  AFTER :  REALLY - thatï¿½s cool  Iï¿½m an Horror Movie Fan too..my best girlfriend gave me the nickname SPOOKY because of that ))
----------------------------------------

[Tweet 3]
  BEFORE:  and  .... YUCKKKKKKKK!! eww.. what is she thinking!? she is soo beautiful and could do SO MUCHHH BETTER!!!!   
  AFTER :  and  .. YUCKK!! eww.. what is she thinking!? she is soo beautiful and could do SO MUCHH BETTER!!  
----------------------------------------

[Tweet 4]
  BEFORE: have heart whaaaaat? 
  AFTER : have heart whaat? 
----------------------------------------

[Tweet 5]
  BEFORE:  haha. im gooooood thanx  but very bored :\ twitter. not facebook. myspace. anything. footy. btw, congrats on the win  hahaha
  AFTER :  haha. im good thanx  but very bored :\ twitter. not facebook. myspace. anything. footy. btw, congra

### 3.6 Replace Emoticons

In [12]:
def replace_emoticon(text):
    for emoticon, meaning in EMOTICONS_EMO.items():
        text = text.replace(emoticon, f" {meaning.replace(" ", "_")} ")
    return text
    
tweets = show_step(tweets, replace_emoticon, "Emoticon Replacement")

 STEP: Emoticon Replacement

[Tweet 0]
  BEFORE: Bought yarn to make daughters "skirt" to wear with leggings. She's fun to make things for ;) I think I'll write this pattern out 
  AFTER : Bought yarn to make daughters "skirt" to wear with leggings. She's fun to make things for  Wink_or_smirk  I think I'll write this pattern out 
----------------------------------------


### 3.7 Remove Numbers

In [13]:
tweets = show_step(tweets, lambda text: re.sub(r"\d+", "", text), "Remove Numbers")

 STEP: Remove Numbers

[Tweet 10]
  BEFORE:  u talking like u know where the fuck i live 1700 walkus ct district heights BITCH do you 
  AFTER :  u talking like u know where the fuck i live  walkus ct district heights BITCH do you 
----------------------------------------


### 3.8 Remove Special Characters

In [14]:
tweets = show_step(tweets, lambda text: re.sub(r"[^a-zA-Z\s!?_\.]", "", text), "Remove Special Characters")

 STEP: Remove Special Characters

[Tweet 0]
  BEFORE: Bought yarn to make daughters "skirt" to wear with leggings. She's fun to make things for  Wink_or_smirk  I think I'll write this pattern out 
  AFTER : Bought yarn to make daughters skirt to wear with leggings. Shes fun to make things for  Wink_or_smirk  I think Ill write this pattern out 
----------------------------------------

[Tweet 1]
  BEFORE:  REALLY - thatï¿½s cool  Iï¿½m an Horror Movie Fan too..my best girlfriend gave me the nickname SPOOKY because of that ))
  AFTER :  REALLY  thats cool  Im an Horror Movie Fan too..my best girlfriend gave me the nickname SPOOKY because of that 
----------------------------------------

[Tweet 2]
  BEFORE: is a perfectionist and has been still working here and there on her Debian layouts, but now it's time for /dev/shower & /dev/cup/tea. 
  AFTER : is a perfectionist and has been still working here and there on her Debian layouts but now its time for devshower  devcuptea. 
-------------

### 3.9 Lowercase and Strip

In [15]:
tweets = show_step(tweets, lambda text: text.lower().strip(), "Lowercase and Strip")

 STEP: Lowercase and Strip

[Tweet 0]
  BEFORE: Bought yarn to make daughters skirt to wear with leggings. Shes fun to make things for  Wink_or_smirk  I think Ill write this pattern out 
  AFTER : bought yarn to make daughters skirt to wear with leggings. shes fun to make things for  wink_or_smirk  i think ill write this pattern out
----------------------------------------

[Tweet 1]
  BEFORE:  REALLY  thats cool  Im an Horror Movie Fan too..my best girlfriend gave me the nickname SPOOKY because of that 
  AFTER : really  thats cool  im an horror movie fan too..my best girlfriend gave me the nickname spooky because of that
----------------------------------------

[Tweet 2]
  BEFORE: is a perfectionist and has been still working here and there on her Debian layouts but now its time for devshower  devcuptea. 
  AFTER : is a perfectionist and has been still working here and there on her debian layouts but now its time for devshower  devcuptea.
----------------------------------------

[T

### 3.10 Standardize Dots

In [16]:
tweets = show_step(tweets, lambda text: re.sub(r"\.{2,}", " ... ", text), "Standardize Dots")

 STEP: Standardize Dots

[Tweet 1]
  BEFORE: really  thats cool  im an horror movie fan too..my best girlfriend gave me the nickname spooky because of that
  AFTER : really  thats cool  im an horror movie fan too ... my best girlfriend gave me the nickname spooky because of that
----------------------------------------

[Tweet 3]
  BEFORE: and  .. yuckk!! eww.. what is she thinking!? she is soo beautiful and could do so muchh better!!
  AFTER : and   ...  yuckk!! eww ...  what is she thinking!? she is soo beautiful and could do so muchh better!!
----------------------------------------

[Tweet 6]
  BEFORE: day off tomorrow so gym shopping and holiday searching  still need a travel buddy so if you know anyone interested.. desperate
  AFTER : day off tomorrow so gym shopping and holiday searching  still need a travel buddy so if you know anyone interested ...  desperate
----------------------------------------

[Tweet 8]
  BEFORE: ci yume n lag vi ..yobanbe th n mt pass..dis..dek c ci qu

### 3.11 Remove Extra Whitespace

In [17]:
tweets = show_step(tweets, lambda text: re.sub(r"\s+", " ", text), "Remove Extra Whitespace")

 STEP: Remove Extra Whitespace

[Tweet 0]
  BEFORE: bought yarn to make daughters skirt to wear with leggings. shes fun to make things for  wink_or_smirk  i think ill write this pattern out
  AFTER : bought yarn to make daughters skirt to wear with leggings. shes fun to make things for wink_or_smirk i think ill write this pattern out
----------------------------------------

[Tweet 1]
  BEFORE: really  thats cool  im an horror movie fan too ... my best girlfriend gave me the nickname spooky because of that
  AFTER : really thats cool im an horror movie fan too ... my best girlfriend gave me the nickname spooky because of that
----------------------------------------

[Tweet 2]
  BEFORE: is a perfectionist and has been still working here and there on her debian layouts but now its time for devshower  devcuptea.
  AFTER : is a perfectionist and has been still working here and there on her debian layouts but now its time for devshower devcuptea.
----------------------------------------

[

### 3.12 Lemmatization and removing stop words

In [18]:
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser", "tok2vec"])

vader_lexicon = SentimentIntensityAnalyzer().lexicon
sentiment_words = set(vader_lexicon.keys())
overlapping_sentiment_stops = nlp.Defaults.stop_words.intersection(sentiment_words)

nlp.Defaults.stop_words -= overlapping_sentiment_stops
for word in overlapping_sentiment_stops:
    nlp.vocab[word].is_stop = False

def lemmatization(text):
    doc = nlp(text)
    text = " ".join([
        token.text if token.is_punct else token.lemma_ 
        for token in doc 
        if (not token.is_stop or token.is_punct) and (token.is_punct or len(token.text) > 1)
    ])
    return text
tweets = show_step(tweets, lemmatization, "Lemmatization")

 STEP: Lemmatization

[Tweet 0]
  BEFORE: bought yarn to make daughters skirt to wear with leggings. shes fun to make things for wink_or_smirk i think ill write this pattern out
  AFTER : bought yarn daughters skirt wear leggings . fun things wink_or_smirk think ill write pattern
----------------------------------------

[Tweet 1]
  BEFORE: really thats cool im an horror movie fan too ... my best girlfriend gave me the nickname spooky because of that
  AFTER : cool horror movie fan ... best girlfriend gave nickname spooky
----------------------------------------

[Tweet 2]
  BEFORE: is a perfectionist and has been still working here and there on her debian layouts but now its time for devshower devcuptea.
  AFTER : perfectionist working debian layouts time devshower devcuptea .
----------------------------------------

[Tweet 3]
  BEFORE: and ... yuckk!! eww ... what is she thinking!? she is soo beautiful and could do so muchh better!!
  AFTER : ... yuckk ! ! eww ... thinking ! ? soo b

### 3.13 Simple Pipeline

In [19]:
# Compile all regex patterns and loading the spacy model
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser", "tok2vec"])

vader_lexicon = SentimentIntensityAnalyzer().lexicon
sentiment_words = set(vader_lexicon.keys())
overlapping_sentiment_stops = nlp.Defaults.stop_words.intersection(sentiment_words)

nlp.Defaults.stop_words -= overlapping_sentiment_stops
for word in overlapping_sentiment_stops:
    nlp.vocab[word].is_stop = False

URL_PATTERN = re.compile(r"http[s]?://\S+|www\.\S+")
MENTION_PATTERN = re.compile(r"@\w+")
HASHTAG_PATTERN = re.compile(r"#")
ELONGATED_PATTERN = re.compile(r"(\w)\1{2,}")
NUMBER_PATTERN = re.compile(r"\d+")
SPECIAL_CHAR_PATTERN = re.compile(r"[^a-zA-Z\s!?_\.]")
WHITESPACE_PATTERN = re.compile(r"\s+")
DOTS_PATTERN = re.compile(r'\.{2,}')

In [20]:
def clean_tweet(text):
    """Single tweet cleaning with preloaded nlp model"""
    # 1. fix html entities
    text = html.unescape(text)
    # 2. remove URLs
    text = URL_PATTERN.sub("", text)
    # 3. remove @mentions
    text = MENTION_PATTERN.sub("", text)
    # 4. remove hashtag symbol but keep word
    text = HASHTAG_PATTERN.sub("", text)
    # 5. cap elongated words
    text = ELONGATED_PATTERN.sub(r"\1\1", text)
    # 6. replace emoticons
    for emoticon, meaning in EMOTICONS_EMO.items():
        if emoticon in text:
            text = text.replace(emoticon, f" {meaning.replace(' ', '_')} ")
    # 7. remove numbers
    text = NUMBER_PATTERN.sub("", text)
    # 8. remove special characters
    text = SPECIAL_CHAR_PATTERN.sub(" ", text)
    # 9. lowercase and strip 
    text = text.lower().strip()
    # 10. turn 2+ dots into exactly 3 (...) and pad with spaces for tokenization
    text = DOTS_PATTERN.sub(" ... ", text)
    # 11. clean up any new double spaces created by the padding
    text = WHITESPACE_PATTERN.sub(" ", text).strip()
    # 12. lemmatization + removing stop words
    doc = nlp(text)
    text = " ".join([
        token.text if token.is_punct else token.lemma_ 
        for token in doc 
        if (not token.is_stop or token.is_punct) and (token.is_punct or len(token.text) > 1)
    ])
    return text

### 3.14 Optimized Pipeline

In [21]:
emoticon_dict = {re.escape(k): f" {v.replace(' ', '_')} " for k, v in EMOTICONS_EMO.items()}
EMOTICON_PATTERN= re.compile("|".join(emoticon_dict.keys()))

vader_lexicon = SentimentIntensityAnalyzer().lexicon
sentiment_words = set(vader_lexicon.keys())

    
def process_batch(texts):
    """Process a batch of tweets in one worker"""
    # Load model once per worker
    nlp = spacy.load("en_core_web_sm", disable=["ner", "parser", "tok2vec"])

    overlapping_sentiment_stops = nlp.Defaults.stop_words.intersection(sentiment_words)
    
    nlp.Defaults.stop_words -= overlapping_sentiment_stops
    for word in overlapping_sentiment_stops:
        nlp.vocab[word].is_stop = False
        
    # Pre-process all texts
    preprocessed = []
    for text in texts:
        text = html.unescape(text)
        text = URL_PATTERN.sub("", text)
        text = MENTION_PATTERN.sub("", text)
        text = HASHTAG_PATTERN.sub("", text)
        text = ELONGATED_PATTERN.sub(r"\1\1", text)
        text = EMOTICON_PATTERN.sub(lambda match: emoticon_dict[re.escape(match.group(0))], text)
        text = NUMBER_PATTERN.sub("", text)
        text = SPECIAL_CHAR_PATTERN.sub(" ", text)
        text = text.lower().strip()
        text = DOTS_PATTERN.sub(" ... ", text)
        preprocessed.append(text)
    
    # Batch lemmatization and stop word remval using nlp.pipe
    inner_batch = min(5000, max(50, len(preprocessed) // 10))
    results = []
    for doc in tqdm(nlp.pipe(preprocessed, batch_size=inner_batch), total=len(preprocessed)):
        text = " ".join([
            token.text if token.is_punct else token.lemma_ 
            for token in doc 
            if (not token.is_stop or token.is_punct) and (token.is_punct or len(token.text) > 1)
        ])
        results.append(WHITESPACE_PATTERN.sub(" ", text))
    
    return results

def clean_tweets_parallel(tweets, n_workers=None, batch_size=5000):
    """Process all tweets in parallel"""
    if n_workers is None:
        n_workers = cpu_count()
    
    # Split into batches
    batches = [tweets[i:i+batch_size] for i in range(0, len(tweets), batch_size)]
    
    print(f"Processing {len(tweets)} tweets in {len(batches)} batches using {n_workers} workers...")
    
    # Process batches in parallel
    with Pool(n_workers) as pool:
        results = pool.map(process_batch, batches)
    
    # Flatten results
    return [item for batch in results for item in batch]

# Usage:
# cleaned = clean_tweets_parallel(df['tweet'].tolist())
# df['cleaned_tweet'] = cleaned

### 3.15 Benchmarking Both Piplelines

In [22]:
sample_size = 100_000
samples = df.sample(sample_size)

In [23]:
s = time.perf_counter()
_ = samples["tweet"].progress_apply(clean_tweet)
e = time.perf_counter()
print(f"Time: {e-s}")
print(f"{sample_size / (e - s):.0f} items/s")

100%|█████████████████████████████████████| 100000/100000 [02:04<00:00, 802.67it/s]

Time: 124.5918814150009
803 items/s


In [24]:
# keep the batch size to like total_size / workers in my case i have i3 5005u so only 2 core and 2 virtual ones
# so it's not that fast in comparision but in mordarn cpus it should be a lot faster 
s = time.perf_counter()
_ = clean_tweets_parallel(samples["tweet"].to_list(), batch_size=25000)
e = time.perf_counter()
print(f"Time: {e-s}")
print(f"{sample_size / (e - s):.0f} items/s")

Processing 100000 tweets in 4 batches using 4 workers...


100%|███████████████████████████████████████| 25000/25000 [00:52<00:00, 474.59it/s]


Time: 56.568276997000794
1768 items/s


## 4. Testing Cleaning Function

### 4.1 Sample Before vs After Comparison

In [25]:
tweet = "I am SO ANGRY right nowwwwww!!!!! This ........ is AMEZING (ToT)"
cleaned = show_step({0:tweet}, clean_tweet, "Example Tweet")

 STEP: Example Tweet

[Tweet 0]
  BEFORE: I am SO ANGRY right nowwwwww!!!!! This ........ is AMEZING (ToT)
  AFTER : angry right noww ! ! ! ! ! ... amezing sad_or_crying
----------------------------------------


In [26]:
cleaned = clean_tweets_parallel([tweet])[0]

print(f"\n[Tweet {0}]")
print(f"  BEFORE: {tweet}")
print(f"  AFTER : {cleaned}")
print("-" * 40)

Processing 1 tweets in 1 batches using 4 workers...


100%|███████████████████████████████████████████████| 1/1 [00:00<00:00, 150.57it/s]



[Tweet 0]
  BEFORE: I am SO ANGRY right nowwwwww!!!!! This ........ is AMEZING (ToT)
  AFTER : angry right noww ! ! ! ! ! ... amezing sad_or_crying
----------------------------------------


In [27]:
cleaned_test_tweets = show_step(test_tweets, clean_tweet, "Cleaning Test Tweets")

 STEP: Cleaning Test Tweets

[Tweet 0]
  BEFORE: Bought yarn to make daughters &quot;skirt&quot; to wear with leggings. She's fun to make things for ;) I think I'll write this pattern out 
  AFTER : bought yarn daughters skirt wear leggings . fun things wink_or_smirk think ll write pattern
----------------------------------------

[Tweet 1]
  BEFORE: @Impala_Guy REALLY - thatï¿½s cool  Iï¿½m an Horror Movie Fan too.....my best girlfriend gave me the nickname SPOOKY because of that ))
  AFTER : cool horror movie fan ... best girlfriend gave nickname spooky
----------------------------------------

[Tweet 2]
  BEFORE: is a perfectionist and has been still working here and there on her #Debian layouts, but now it's time for /dev/shower &amp; /dev/cup/tea. 
  AFTER : perfectionist working debian layouts time dev shower dev cup tea .
----------------------------------------

[Tweet 3]
  BEFORE: @ddlovato and @TraceCyrus .... YUCKKKKKKKK!! eww.. what is she thinking!? she is soo beautiful an

In [28]:
keys = [key for key in test_tweets]
tweets = [test_tweets[key] for key in test_tweets]
cleaned_tweets = clean_tweets_parallel(tweets)

for idx, cleaned in zip(keys, cleaned_tweets):
    print(f"\n[Tweet {idx}]")
    print(f"  BEFORE: {tweets[idx]}")
    print(f"  AFTER : {cleaned}")
    print("-" * 40)

Processing 11 tweets in 1 batches using 4 workers...


100%|█████████████████████████████████████████████| 11/11 [00:00<00:00, 244.25it/s]



[Tweet 0]
  BEFORE: Bought yarn to make daughters &quot;skirt&quot; to wear with leggings. She's fun to make things for ;) I think I'll write this pattern out 
  AFTER : bought yarn daughters skirt wear leggings . fun things wink_or_smirk think ll write pattern
----------------------------------------

[Tweet 1]
  BEFORE: @Impala_Guy REALLY - thatï¿½s cool  Iï¿½m an Horror Movie Fan too.....my best girlfriend gave me the nickname SPOOKY because of that ))
  AFTER :  cool horror movie fan ... best girlfriend gave nickname spooky
----------------------------------------

[Tweet 2]
  BEFORE: is a perfectionist and has been still working here and there on her #Debian layouts, but now it's time for /dev/shower &amp; /dev/cup/tea. 
  AFTER : perfectionist working debian layouts time dev shower dev cup tea .
----------------------------------------

[Tweet 3]
  BEFORE: @ddlovato and @TraceCyrus .... YUCKKKKKKKK!! eww.. what is she thinking!? she is soo beautiful and could do SO MUCHHH BETTER

### 4.2 Cleaning Random Samples

In [29]:
sample_size = 5
sample_tweets = {idx:tweet for idx, tweet in enumerate(df["tweet"].sample(sample_size))}

In [30]:
clean_tweets = show_step(sample_tweets, clean_tweet, f"Cleaning {sample_size} Random Tweets")

 STEP: Cleaning 5 Random Tweets

[Tweet 0]
  BEFORE: last day in longboat 
  AFTER : day longboat
----------------------------------------

[Tweet 1]
  BEFORE: I've just checked my bank balance, oh man it's a sorry state 
  AFTER : ve checked bank balance oh man sorry state
----------------------------------------

[Tweet 2]
  BEFORE: @dearcandy It's the hardest thing to get started sometimes, but boy, when it's done it feels marvellous! Can't wait to feel marvellous 
  AFTER : hardest thing started boy feels marvellous ! wait feel marvellous
----------------------------------------

[Tweet 3]
  BEFORE: Last day of school... but probably pulling an all-nighter after graduation... 
  AFTER : day school ... probably pulling nighter graduation ...
----------------------------------------

[Tweet 4]
  BEFORE: @TreoBenny ha! I know..I'm very ashamed. 
  AFTER : ha ! know ... ashamed .
----------------------------------------


In [31]:
keys = [key for key in sample_tweets]
tweets = [sample_tweets[key] for key in sample_tweets]
cleaned_tweets = clean_tweets_parallel(tweets)

for idx, cleaned in zip(keys, cleaned_tweets):
    print(f"\n[Tweet {idx}]")
    print(f"  BEFORE: {tweets[idx]}")
    print(f"  AFTER : {cleaned}")
    print("-" * 40)

Processing 5 tweets in 1 batches using 4 workers...


100%|███████████████████████████████████████████████| 5/5 [00:00<00:00, 577.86it/s]



[Tweet 0]
  BEFORE: last day in longboat 
  AFTER : day longboat
----------------------------------------

[Tweet 1]
  BEFORE: I've just checked my bank balance, oh man it's a sorry state 
  AFTER : ve checked bank balance oh man sorry state
----------------------------------------

[Tweet 2]
  BEFORE: @dearcandy It's the hardest thing to get started sometimes, but boy, when it's done it feels marvellous! Can't wait to feel marvellous 
  AFTER : hardest thing started boy feels marvellous ! wait feel marvellous
----------------------------------------

[Tweet 3]
  BEFORE: Last day of school... but probably pulling an all-nighter after graduation... 
  AFTER : day school ... probably pulling nighter graduation ...
----------------------------------------

[Tweet 4]
  BEFORE: @TreoBenny ha! I know..I'm very ashamed. 
  AFTER : ha ! know ... ashamed .
----------------------------------------


## 5. Apply Cleaning to Full Dataset

In [32]:
df["cleaned_tweet"] = clean_tweets_parallel(df["tweet"].to_list(), batch_size=400_000)

Processing 1600000 tweets in 4 batches using 4 workers...


100%|█████████████████████████████████████| 400000/400000 [14:37<00:00, 455.89it/s]


## 6. Post-Cleaning Validation

### 6.1 Check for Nulls

In [33]:
df.isnull().sum()

sentiment        0
id               0
tweet            0
cleaned_tweet    0
dtype: int64

### 6.2 Check for Empty Strings

In [34]:
(df["cleaned_tweet"] == "").sum()

6768

In [35]:
df = df[df["cleaned_tweet"] != ""]

In [36]:
(df["cleaned_tweet"] == "").sum()

0

### 6.3 Class Balance After Cleaning

In [37]:
df["sentiment"].value_counts()

sentiment
4    796628
0    796604
Name: count, dtype: int64

### 6.4 Sample Cleaned Tweets

In [38]:
df["cleaned_tweet"].sample(5)

1282360                      oo love soft spots good pillows
914299                                           thanks bb .
1425283                               hmm . mind number time
751261     sis away long weird room tings no talk mis leonda
166269                                              want goo
Name: cleaned_tweet, dtype: str

## 7. Save Cleaned Dataset

In [39]:
df.to_csv("../data/processed/cleaned_tweets_for_ml.csv", index=False)